In [ ]:
import numpy as np
import math
from decimal import Decimal, getcontext
from functools import lru_cache

getcontext().prec = 150

@lru_cache(None)
def factorial_dec(n: int) -> Decimal:
    if n < 0:
        raise ValueError("Negative factorial not defined")
    if n <= 1:
        return Decimal(1)
    return Decimal(n) * factorial_dec(n - 1)

@lru_cache(None)
def comb_dec(n: int, k: int) -> Decimal:
    if k < 0 or k > n:
        return Decimal(0)
    return factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

@lru_cache(None)
def D_coef(n: int, j: int) -> Decimal:
    if j == 0:
        return Decimal(1)
    
    result = Decimal(0)
    for k in range(0, n + 2*(j-1) + 1):
        for l in range(0, j):
            term = (Decimal((-1)**k) * 
                   D_coef(n, l) * 
                   comb_dec(n + 2*l, k - (j-1) + l) * 
                   Decimal((n + 2*(j-1) - 2*k) ** (n + 2*(j-1) + 2)) / 
                   (Decimal(2) ** (n + 2*(j-1) + 2) * 
                    factorial_dec(n + 2*(j-1) + 2)))
            result += term
    return result

@lru_cache(None)
def A_coef(k: int, n: int, m: int) -> Decimal:
    if k < 0 or k > 2*m:
        return Decimal(0)
    
    result = Decimal(0)
    sign = Decimal((-1)**(k - m))
    
    for l in range(0, m + 1):
        comb_arg = k - m + l
        if comb_arg >= 0:
            term = (sign * 
                   D_coef(n, l) * 
                   comb_dec(n + 2*l, comb_arg))
            result += term
    
    return result

@lru_cache(None)
def W_coef(m: int, k: int) -> float:
    result = Decimal(0)
    for n in range(0, m + 1):
        if m - n >= 0:
            a_val = A_coef(k, 2*n, m - n)
            denom = (Decimal(2) ** (2*n)) * factorial_dec(2*n + 1)
            result += a_val / denom
    return float(result)

def get_weights(m: int):
    weights = []
    for k in range(-m, m + 1):
        idx = m - k
        weights.append(W_coef(m, idx))
    return np.array(weights)

def integrate_diff_scheme(f, a, b, J, m):
    h = (b - a) / J
    weights = get_weights(m)
    offset = m
    
    y_values = []
    for j in range(-m, J + m):
        x = a + (j + 0.5) * h
        y_values.append(f(x))
    
    y = np.array(y_values)
    
    integral = 0.0
    for j in range(J):
        idx_start = j + offset
        y_slice = y[idx_start - m:idx_start + m + 1]
        integral += np.sum(weights * y_slice)
    
    return integral * h

def integrand(x):
    x = float(x)
    
    if abs(x) < 1e-6:
        return 0.5 - x/4 + x*x/24
    
    return math.sin(x/2) / (math.exp(x) - 1)

def compute_integral_machine_precision():
    a, b = -1.0, 1.0
    
    m = 18

    J = 512       
    
    I = integrate_diff_scheme(integrand, a, b, J, m)
    
    I_half = integrate_diff_scheme(integrand, a, b, J//2, m)
    p = 2*m + 2
    error_est = abs(I - I_half) / (2**p - 1)
    
    return I, error_est

I, error = compute_integral_machine_precision()

print(f"{I:.11f}")
print(error)

1.01303923622
2.2335492126146896e-24


In [39]:
def compute_T1() -> float:
    return math.pi**2 / 6 - math.pi + 1


def compute_S2(N: int) -> float:

    total = 0.0
    for n in range(1, N + 1):
        total += math.cos(2 * n) / (n**2 * (n**4 + n**2 + 1))
    return total


def tail_bound(N: int) -> float:

    return 1.0 / (5 * N**5)


def find_N(eps: float) -> int:

    N = 1
    while tail_bound(N) >= eps:
        N += 1
    return N


def kummer_sum(eps: float = 1e-6) -> float:

    T1 = compute_T1()
    N = find_N(eps)
    S2 = compute_S2(N)
    S = T1 - S2

    return S


result = kummer_sum(eps=1e-8)
print(result)

-0.35126538649088485


In [ ]:
def y(x):
    if x > 0:
        return math.cosh(math.sqrt(x))
    elif x < 0:
        return math.cos(math.sqrt(-x))
    else:
        return 1.0


def g(x):

    if abs(x) < 1e-30:
        return 0.5
    return (y(x) - 1.0) / x


def f(x):
    if abs(x) < 1e-300:
        return 0.5
    return (g(x) + g(-x)) / 2.0


def lagrange_at_zero(xs, ys):
    n = len(xs)
    result = 0.0
    for i in range(n):
        L = 1.0
        for j in range(n):
            if j != i:
                L *= (0.0 - xs[j]) / (xs[i] - xs[j])
        result += ys[i] * L
    return result

def compute_limit(dx, m):
    j0 = 2 * m + 2
    xs = [(2*j - 1) * dx / 2.0 for j in range(1, j0 + 1)]
    ys = [f(x) for x in xs]
    return lagrange_at_zero(xs, ys)

EXACT = 0.5

for h in [1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12, 1e-14]:
    val = g(h)
    print(f"  {h:10.2e}  {val:20.15f}  {abs(val - EXACT):12.2e}")

    1.00e-01     0.504180580384721      4.18e-03
    1.00e-02     0.500416805580350      4.17e-04
    1.00e-04     0.500004166681389      4.17e-06
    1.00e-06     0.500000041592230      4.16e-08
    1.00e-08     0.499999996961265      3.04e-09
    1.00e-10     0.500000041370185      4.14e-08
    1.00e-12     0.500044450291171      4.45e-05
    1.00e-14     0.510702591327572      1.07e-02
